# 05 Save and Load

This notebook audits TorchLens persistence: `tl.save`, `tl.load`, the portable `.tlspec` directory bundle, and which fields survive a round trip. A `.tlspec` bundle is the on-disk folder format TorchLens uses for trace data and tensor blobs.

We use a tiny deterministic model and write generated bundles under `notebooks/audit/_artifacts/` so tutorial outputs stay separate from source notebooks.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "audit":
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo root: {REPO_ROOT}")

In [ ]:
import json
import shutil
from pathlib import Path

import torch
from torch import nn
import torchlens as tl

ARTIFACT_DIR = REPO_ROOT / "notebooks" / "audit" / "_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(23)


class SaveLoadNet(nn.Module):
    """Tiny model used for `.tlspec` save/load checks."""

    def __init__(self) -> None:
        """Initialize layers."""

        super().__init__()
        self.stem = nn.Linear(3, 4)
        self.head = nn.Linear(4, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run the model.

        Parameters
        ----------
        x:
            Input tensor.

        Returns
        -------
        torch.Tensor
            Two-column model output.
        """

        hidden = torch.relu(self.stem(x))
        return self.head(hidden)


model = SaveLoadNet().eval()
x = torch.tensor([[0.25, -0.5, 1.0]])
trace = tl.trace(
    model,
    x,
    save_arg_values=True,
    save_rng_states=True,
    random_seed=23,
)
print(f"trace type: {type(trace).__name__}")
print(f"layers: {trace.layer_labels}")
print(f"output layer: {trace.output_layers[0]}")

`tl.save(trace, path)` writes a portable `.tlspec` directory. The manifest is JSON; larger tensor payloads live in a `blobs/` subdirectory.

In [ ]:
bundle_path = ARTIFACT_DIR / "05_trace_roundtrip.tlspec"
if bundle_path.exists():
    shutil.rmtree(bundle_path)

tl.save(
    trace,
    bundle_path,
    level="portable",
    include_outs=True,
    include_grads=True,
    include_saved_args=True,
    include_rng_states=True,
    overwrite=True,
)

print(f"saved bundle: {bundle_path}")
print(f"top-level files: {sorted(path.name for path in bundle_path.iterdir())}")
print(f"blob files: {len(list((bundle_path / 'blobs').iterdir()))}")

The manifest records format metadata, counts, tensor entries, and save level. This is the quick way to inspect a bundle without loading every tensor.

In [ ]:
manifest = json.loads((bundle_path / "manifest.json").read_text())
for key in ["kind", "tlspec_version", "bundle_format", "save_level", "n_layers"]:
    print(f"{key}: {manifest[key]}")
print(f"tensor entries: {len(manifest['tensors'])}")
print(f"unsupported tensors: {len(manifest['unsupported_tensors'])}")

`tl.load(path)` restores a `Trace`. Stable labels, shapes, saved activations, and selected metadata survive the round trip.

In [ ]:
loaded = tl.load(bundle_path)
print(f"loaded type: {type(loaded).__name__}")
print(f"loaded from bundle: {getattr(loaded, '_loaded_from_bundle', False)}")
print(f"same layer labels: {loaded.layer_labels == trace.layer_labels}")
print(f"same output labels: {loaded.output_layers == trace.output_layers}")

label = trace.output_layers[0]
original_out = trace[label].out
loaded_out = loaded[label].out
print(f"output shape after load: {tuple(loaded_out.shape)}")
print(f"output values match: {torch.allclose(original_out, loaded_out)}")

Portable bundles preserve public trace data, not live Python runtime links. The restored trace is for inspection and portable data exchange; the original model object is not re-created from the bundle.

In [ ]:
field_names = [
    "model_class_name",
    "backend",
    "num_layers",
    "num_saved_layers",
    "saved_activation_memory_str",
]
for field_name in field_names:
    print(f"{field_name}: {getattr(loaded, field_name)}")

source_ref = getattr(trace, "_source_model_ref", None)
loaded_ref = getattr(loaded, "_source_model_ref", None)
original_source = source_ref() if source_ref is not None else None
loaded_source = loaded_ref() if loaded_ref is not None else None
print(f"source model still live on original: {original_source is model}")
print(f"loaded source model is original object: {loaded_source is model}")

`level="audit"` is the more permissive save level for local inspection. Portable is the safer default for sharing; audit-level data can include more local-only metadata when the trace contains it.

Bookkeeping note: `n_layers` counts trace records, while manifest `tensors` counts serialized tensor blobs. `include_saved_args` and save level can change tensor blob counts without changing layer counts.

In [ ]:
audit_path = ARTIFACT_DIR / "05_trace_audit.tlspec"
if audit_path.exists():
    shutil.rmtree(audit_path)

tl.save(trace, audit_path, level="audit", overwrite=True)
audit_manifest = json.loads((audit_path / "manifest.json").read_text())
print(f"audit save level: {audit_manifest['save_level']}")
print(f"audit tensor entries: {len(audit_manifest['tensors'])}")
print(f"audit bundle files: {sorted(path.name for path in audit_path.iterdir())}")

> NOTE: The current executable `.tlspec` trace format is a directory bundle with `manifest.json`, `metadata.pkl`, and `blobs/`. Treat the folder as the saved object path rather than expecting a single flat file.

## 🔧 Sandbox

Mini-experiment: change `sandbox_level`, `sandbox_include_args`, and `sandbox_overwrite`. Expected observation: layer counts stay stable, while manifest tensor entries can move when argument blobs are included or excluded.

In [ ]:
sandbox_level = "portable"
sandbox_include_args = True
sandbox_overwrite = True
sandbox_path = ARTIFACT_DIR / "05_sandbox_trace.tlspec"
if sandbox_path.exists():
    shutil.rmtree(sandbox_path)

tl.save(
    trace,
    sandbox_path,
    level=sandbox_level,
    include_saved_args=sandbox_include_args,
    overwrite=sandbox_overwrite,
)
sandbox_manifest = json.loads((sandbox_path / "manifest.json").read_text())
print(f"level: {sandbox_manifest['save_level']}")
print(f"layers: {manifest['n_layers']} -> {sandbox_manifest['n_layers']}")
print(f"tensor entries: {len(manifest['tensors'])} -> {len(sandbox_manifest['tensors'])}")
print(f"include_saved_args: {sandbox_include_args}")